# PGR Safety Experiments

Safety-Aware Prioritized Generative Replay: measuring and fixing hazard compression in diffusion-based replay.

**Before running:** Change runtime to **GPU (A100 preferred)** via Runtime > Change runtime type.

## 1. Mount Google Drive & Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_DIR = '/content/pgr'
DRIVE_SAFETY = '/content/drive/MyDrive/pgr/safety'
RESULTS_DIR = '/content/drive/MyDrive/pgr/safety_results'

# Clone PGR repo (only if not already there)
if not os.path.exists(REPO_DIR):
    !git clone --recursive https://github.com/renwang435/pgr.git {REPO_DIR}
else:
    print(f'{REPO_DIR} already exists, skipping clone')

# Copy our safety/ extension from Google Drive
!mkdir -p {REPO_DIR}/safety
!cp -r {DRIVE_SAFETY}/* {REPO_DIR}/safety/
print('Copied safety/ from Google Drive')

# Create results directory on Google Drive
!mkdir -p {RESULTS_DIR}

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

## 2. Install Dependencies

In [ ]:
!python safety/colab_setup.py

In [ ]:
# Verify GPU
import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU! Change runtime to GPU (A100).')

## 3. Smoke Test (2 epochs, quick sanity check)

In [ ]:
!python safety/online_cost_cond.py \
    --env cheetah-run-v0 \
    --seed 42 \
    --gin_config_files config/online/sac.gin \
    --gin_params "redq_sac.epochs = 2" "redq_sac.disable_diffusion = True" \
    --velocity_threshold 7.0 \
    --sac_only \
    --results_folder {RESULTS_DIR}/smoke_test

In [ ]:
# Check that results were saved
import json, glob
smoke_files = glob.glob(f'{RESULTS_DIR}/smoke_test/*.json')
print(f'Smoke test files: {smoke_files}')
if smoke_files:
    with open(smoke_files[0]) as f:
        data = json.load(f)
    print(f"Episodes: {len(data['episode_rewards'])}, Complete: {data.get('complete', 'N/A')}")
    print('Smoke test PASSED')
else:
    print('Smoke test FAILED - no results saved!')

## 4. Full Experiments (3 methods x 3 seeds)

Each run takes ~30-90 min on A100. Total: ~5-14 hours.

Results are saved to Google Drive after every epoch, so you won't lose progress if the runtime disconnects.

### 4a. SAC Baseline (3 seeds)

In [ ]:
ENV = 'cheetah-run-v0'
VEL = 7.0
SEEDS = [42, 123, 456]

In [ ]:
for seed in SEEDS:
    print(f'\n{"="*60}')
    print(f'SAC baseline, seed={seed}')
    print(f'{"="*60}')
    !python safety/online_cost_cond.py \
        --env {ENV} \
        --seed {seed} \
        --gin_config_files config/online/sac.gin \
        --gin_params "redq_sac.disable_diffusion = True" \
        --velocity_threshold {VEL} \
        --sac_only \
        --results_folder {RESULTS_DIR}

### 4b. PGR (3 seeds)

In [ ]:
for seed in SEEDS:
    print(f'\n{"="*60}')
    print(f'PGR, seed={seed}')
    print(f'{"="*60}')
    !python safety/online_cost_cond.py \
        --env {ENV} \
        --seed {seed} \
        --gin_config_files config/online/sac_cond_synther_dmc.gin \
        --gin_params "redq_sac.cond_top_frac = 0.25" \
        --velocity_threshold {VEL} \
        --results_folder {RESULTS_DIR}

### 4c. PGR+Memory — Our Contribution (3 seeds)

In [ ]:
for seed in SEEDS:
    print(f'\n{"="*60}')
    print(f'PGR+Memory, seed={seed}')
    print(f'{"="*60}')
    !python safety/online_cost_cond.py \
        --env {ENV} \
        --seed {seed} \
        --gin_config_files config/online/sac_cond_synther_dmc.gin \
        --gin_params "redq_sac.cond_top_frac = 0.25" \
        --velocity_threshold {VEL} \
        --use_rare_buffer \
        --use_lagrangian \
        --results_folder {RESULTS_DIR}

## 5. Check Results

In [ ]:
import json, glob

result_files = sorted(glob.glob(f'{RESULTS_DIR}/*.json'))
print(f'Found {len(result_files)} result files:\n')
for f in result_files:
    with open(f) as fh:
        d = json.load(fh)
    n_eps = len(d.get('episode_rewards', []))
    complete = d.get('complete', 'N/A')
    last_rew = d['episode_rewards'][-1] if d.get('episode_rewards') else 'N/A'
    print(f'  {os.path.basename(f):40s}  eps={n_eps:4d}  complete={complete}  last_reward={last_rew}')

## 6. Generate Figures

In [ ]:
# Generate figures to Google Drive
FIGURES_DIR = '/content/drive/MyDrive/pgr/figures'
!python safety/make_figures.py --results_dir {RESULTS_DIR} --output_dir {FIGURES_DIR}

print(f'\nFigures saved to {FIGURES_DIR}')
!ls -la {FIGURES_DIR}/

In [ ]:
# Display figures inline
from IPython.display import display, Image
import glob

for fig_path in sorted(glob.glob(f'{FIGURES_DIR}/*.pdf')):
    # Convert PDF to PNG for inline display
    png_path = fig_path.replace('.pdf', '.png')
    !convert -density 150 {fig_path} {png_path} 2>/dev/null || true
    if os.path.exists(png_path):
        print(f'\n{os.path.basename(fig_path)}:')
        display(Image(png_path))
    else:
        print(f'{os.path.basename(fig_path)} saved (install imagemagick to display inline)')